## varying white noise  find the systematic this is related to

#### Importing Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sbi.utils import BoxUniform
from torch.distributions import LogNormal, Independent
from joblib import Parallel, delayed
from sbi.analysis import pairplot
from sbi.inference import NPE
from sbi.analysis import plot_summary
import matplotlib.pyplot as plt
import camb
import healpy as hp

_ = torch.manual_seed(42)

_ = np.random.seed(0)

/home/vasir/cmb-work/.venv/lib/python3.12/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


#### Defining Functions

In [2]:
def get_camb_spectrum(theta, lmax=3000):
    """Runs CAMB physics engine for a specific set of parameters."""
    H0_val, ombh2_val, omch2_val, As_val, ns_val = theta
    
    pars = camb.set_params(
        H0=H0_val, ombh2=ombh2_val, omch2=omch2_val, 
        mnu=0.06, omk=0, tau=0.06, 
        As=As_val, ns=ns_val, 
        halofit_version='mead', lmax=lmax
    )
    
    # get c_ell
    results = camb.get_results(pars)
    powers = results.get_cmb_power_spectra(pars, CMB_unit='muK', raw_cl=True)['total']
    
    # Return just the TT spectrum
    return powers[:, 0]

def generateMock(cl, nls=None, lmax=3000, beam_fwhm=5.0):
    """Turns a pure C_ell spectrum into a noisy 2D sky map observation."""
    # Ensure C_ell is the correct length for lmax
    if len(cl) < lmax + 1:
        cl = np.pad(cl, (0, lmax + 1 - len(cl)))
    cl = cl[:lmax+1]
    
    # 1. Generate the True Sky (Cosmic Variance)
    cmb_alm = hp.synalm(cl, lmax=lmax) 
    
    # 2. Apply Telescope Blur
    if beam_fwhm is not None and beam_fwhm != 0:
        beam = hp.gauss_beam(beam_fwhm / 60 / 180 * np.pi, lmax=lmax) 
    else:
        beam = np.ones(lmax + 1)
        
    # 3. Add Telescope Noise (The Splits)
    if nls is not None:
        obs_alms = []
        for nl in nls:
            # Safely format the noise array to match lmax
            nl_formatted = nl[:lmax+1] if len(nl) >= lmax+1 else np.pad(nl, (0, lmax+1-len(nl)))
            
            noise = hp.synalm(nl_formatted, lmax=lmax) 
            obs_alms.append(hp.almxfl(cmb_alm, beam) + noise) 
        return np.array(obs_alms)
    else:
        return hp.almxfl(cmb_alm, beam) if beam_fwhm != 0 else cmb_alm

def getCompression(param_dict,derivatives,beam_fwhm=None,noise_cl=None):
    """ 
    A simple implementation of the moped algorithm. This assumes no dependence of cov mat on parameters (e.g. https://arxiv.org/pdf/1204.4724)
    Compute derivatives numerically
    Compute the cov mat based on a gaussian formalua. 
    Compressor is then \nabla \mu C^{-1} (d-\mu). 
    Note: Match the beam and noise to the splits above.
    This is a close to optimal compression around the fiducial cosmology. Maybe use an SBI prior is some region around here. Planck +3 sigma?
    """
    
    lmax=param_dict['lmax']
    def getSpectrum(params): # similar function to generateMock, adds some noise
        pars = camb.set_params(**params)
        results = camb.get_results(pars)
        powers = results.get_cmb_power_spectra(pars, CMB_unit='muK',raw_cl=True)['total'][2:3001,0]
        if noise_cl is None:
            noise=0
        else:
            noise=noise_cl[2:3001]
        if beam_fwhm is not None:
            beam = hp.gauss_beam(beam_fwhm/60/180*np.pi,lmax=lmax)[2:]
        else:
            beam = np.ones(3001)[2:]
    
        return powers*beam**2+noise # Add in the effect of beam and noise if you want them
    fiducial = getSpectrum(param_dict) # run simulation once to obtain "expected" observation
    cov_mat = 2./(2*np.arange(2,3001)+1)*fiducial**2 # Gaussian cov mat.: error dominated by cosmic variance (have one universe to look at, 2/(2l+1)*C_ell^2)
    derivs = []
    for deriv_name in derivatives.keys():
        step = param_dict[deriv_name]*derivatives[deriv_name]
        deriv_params = dict(param_dict)
        deriv_params[deriv_name]+=step
        up = getSpectrum(deriv_params)
        deriv_params = dict(param_dict)
        deriv_params[deriv_name]-=step
        down = getSpectrum(deriv_params)
        if deriv_name=='As': step*=1e10 # Different "units" for this to adjust ranges of scales (measuring As 10^10 instead of As)
        derivs.append((up-down)/(2*step)) # we calculate gradients by finding difference beteeen slight nudge up and down
        print(deriv_name)
    derivs = np.array(derivs)
    def compressor(data):
        data = data[2:]# Remove first two elements, monopole and dipoel
        return  np.dot(derivs,(data-fiducial)/cov_mat) 
    return fiducial,cov_mat,derivs,compressor

def blanket_simulator(theta, compressor, nl_split, lmax=3000, beam_fwhm=5.0, seed=None):
    
    #takes seed to ensure parallel workers don't duplicate noise
    if seed is not None:
        np.random.seed(seed)
    
    # Get pure physics
    cl_pure = get_camb_spectrum(theta, lmax=lmax)
    
    # Get messy telescope map
    alms = generateMock(cl_pure, nls=[nl_split], lmax=lmax, beam_fwhm=beam_fwhm)
    
    # Extract the noisy observation
    cl_obs = hp.alm2cl(alms[0], lmax=lmax)
    
    # COMPRESS! (Using the function you returned from getCompression)
    x_summary = compressor(cl_obs)
    
    return x_summary


def define_uniform_prior():
    low_bounds  = torch.tensor([60.0, 0.020, 0.10, 1.5e-9, 0.90])
    high_bounds = torch.tensor([75.0, 0.025, 0.14, 2.5e-9, 1.02])
    prior = BoxUniform(low=low_bounds, high=high_bounds)
    return prior

def generate_theta(prior, num_simulations):
    theta = prior.sample((num_simulations,))
    return theta

def parallel_simulate(theta, seeds,compressor, nl_base):
    # Our simulator uses numpy, but prior samples are in PyTorch.
    
    num_workers = 4
    x = Parallel(n_jobs=num_workers)(
    delayed(blanket_simulator)(theta_val.numpy(), compressor, nl_base, seed=s) 
    for theta_val, s in zip(theta, seeds) # 
)
    return torch.tensor(np.array(x), dtype=torch.float32)

<>:49: SyntaxWarning: invalid escape sequence '\m'
<>:49: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_12061/523005499.py:49: SyntaxWarning: invalid escape sequence '\m'
  """


In [ ]:
def train_net_generate_samples(x,theta,x_obs, prior, verbose, max_epoch,true_val):
    
    inference = NPE(prior= prior, density_estimator="nsf")
    posterior_net = inference.append_simulations(theta, x).train(max_num_epochs=max_epoch, stop_after_epochs=30)
    posterior_direct = inference.build_posterior(density_estimator=posterior_net,sample_with="direct")
    posterior_mcmc = inference.build_posterior(density_estimator=posterior_net,sample_with="mcmc")
    samples = posterior_mcmc.sample((1_000,), x=x_obs)

    if verbose:
        _ = plot_summary(
        inference,
        tags=["training_loss", "validation_loss"],
        figsize=(10, 2),
        )

        print(posterior_mcmc)
        print("Observation: ", x_obs)

        _ = pairplot(
            samples,
            points = true_val[None,:],
            limits=[[60.0, 75.0], [0.020, 0.025], [0.10, 0.14], [1.5e-9, 2.5e-9], [0.90, 1.02]],
            ticks=[[60.0, 75.0], [0.020, 0.025], [0.10, 0.14], [1.5e-9, 2.5e-9], [0.90, 1.02]],
            figsize=(10, 10),
            labels=[r"$H_0$", r"$\Omega_b h^2$", r"$\Omega_c h^2$", r"$A_s$", r"$n_s$"]
            )
        
    return samples, posterior_mcmc, posterior_direct, inference

#### Define Fiducial Baseline

In [ ]:
# Define your Baseline Universe and Step Sizes
fiducial_params = {
    'H0': 67.5, 'ombh2': 0.022, 'omch2': 0.122, 
    'As': 2e-9, 'ns': 0.965, 'lmax': 3000
}
derivatives_frac = {
    'H0': 0.001, 'ombh2': 0.005, 'omch2': 0.005, 'As': 0.005, 'ns': 0.005
}

# Define baseline noise array
nl_base = np.ones(3001) * (20/60/180*np.pi)**2
nl_systematic = nl_base * 1.2

# Build Compressor
fiducial, cov, derivs, my_compressor = getCompression(
    param_dict=fiducial_params, 
    derivatives=derivatives_frac,
    beam_fwhm=5.0, 
    noise_cl=nl_base  # standard white noise array
)

In [ ]:
# Define true universe
theta_true = torch.tensor([67.5, 0.022, 0.122, 2e-9, 0.965])
num_simulations = 1_000
prior = define_uniform_prior()
theta = generate_theta(prior, num_simulations)

unique_seeds = np.random.randint(0, 1000000, size=num_simulations) # understand clearly why this step

# Data split 1
x_obs_1 = torch.tensor(blanket_simulator(theta_true, my_compressor, nl_base), dtype=torch.float32)
 ## add something to plot realisation
x_train_1 = parallel_simulate(theta, unique_seeds ,my_compressor, nl_base)

# Data split 2
x_obs_2 = torch.tensor(blanket_simulator(theta_true, my_compressor, nl_systematic), dtype=torch.float32)
 ## add something to plot realisation
x_train_2 = parallel_simulate(theta, unique_seeds ,my_compressor, nl_systematic)


In [ ]:
samples1, posterior1_mcmc, posterior1_direct, inference1 = train_net_generate_samples(x_train_1,theta,x_obs_1, prior, verbose = True, max_epoch=50, true_val=theta_true)

In [ ]:
samples2, posterior2_mcmc, posterior2_direct, inference2 = train_net_generate_samples(x_train_2,theta,x_obs_2, prior, verbose = True, max_epoch=50, true_val=theta_true)